In [1]:
import numpy as np
import os

X = np.load(".data/data.npy")
if os.path.exists(".data/permutation.npy"):
    X_permutation = np.load(".data/permutation.npy")
else:
    rs = np.random.RandomState(42)
    X_permutation = rs.permutation(X.shape[0])
    del rs
    np.save(".data/permutation.npy", X_permutation)
A = X[X_permutation[:8000]]
B = X[X_permutation[8000:10000]]
A_1 = A.T @ A
A_2 = A @ A.T

### Estimativa de tempo para função `eig` vs `svd`

In [ ]:
import matplotlib.pyplot as plt

# Estudos feito em outro notebook para não gastar memória nesse
x = [500, 1000, 2000, 3000, 4000, 5000]
y_eig = [0.3, 2.0, 6.7, 15.6, 29.1, 48.8]
y_svd = [0.1, 0.5, 2.9, 12.1, 24.1, 44.3]
p_eig = np.poly1d(np.polyfit(x, y_eig, 3))
p_svd = np.poly1d(np.polyfit(x, y_svd, 3))     

x_f = np.linspace(1, 30000, 1000)
fig = plt.figure(dpi=300)
plt.plot(x_f, p_eig(x_f), alpha=0.5, label="Projeção np.linalg.eig")
plt.plot(x_f, p_svd(x_f), alpha=0.5, label="Projeção np.linalg.svd")
plt.xlabel("Tamanho da matriz")
plt.ylabel("Duração em segundos")
plt.title("Dimensionalidade x Tempo de execução")
plt.legend()
plt.savefig("comparison_eig_svd_time.png")
plt.show()

In [3]:
import os

if os.path.exists(".data/eigvals_A1.npy"):
    eigvals_A1 = np.load(".data/eigvals_A1.npy")
    eigvals_A2 = np.load(".data/eigvals_A2.npy")
    singvals_A = np.load(".data/singvals_A.npy")
    singvals_A_squared = singvals_A ** 2
else:
    eigvals_A1 = np.linalg.eigvals(A_1)
    np.save(".data/eigvals_A1.npy", eigvals_A1)
    eigvals_A2 = np.linalg.eigvals(A_2)
    np.save(".data/eigvals_A2.npy", eigvals_A2)
    singvals_A = np.linalg.svdvals(A)
    np.save(".data/singvals_A.npy", singvals_A)
    singvals_A_squared = singvals_A ** 2

### Energia Cumulativa

In [4]:
cummulative_A_1 = np.cumsum(eigvals_A1) / np.sum(eigvals_A1) * 100
cummulative_A_2 = np.cumsum(eigvals_A2) / np.sum(eigvals_A2) * 100
cummulative_A = np.cumsum(singvals_A_squared) / np.sum(singvals_A_squared) * 100

In [ ]:
import matplotlib as mpl

mpl.rcParams["text.usetex"] = True

fig = plt.figure(dpi=300)
plt.plot(cummulative_A_1, '-b', label="$\sum_i^k \lambda_1$")
plt.plot(cummulative_A_2, '--r', label="$\sum_i^k \lambda_2$")
plt.plot(cummulative_A, '-.g', label="$\sum_i^k \sigma^2$")
plt.xlabel("Quantidade de modos")
plt.ylabel("Percentual da soma dos autovalores")
plt.legend()
plt.title("Energia cumulativa")
plt.savefig("comparativo_cummulative_energy.png", dpi=300)

In [6]:
if os.path.exists(".data/svd_a_u.npy"):
    U_A, S_A, V_A = (
        np.load(".data/svd_a_u.npy"),
        np.load(".data/svd_a_s.npy"),
        np.load(".data/svd_a_v.npy"),
    )
    U_B, S_B, V_B = (
        np.load(".data/svd_b_u.npy"),
        np.load(".data/svd_b_s.npy"),
        np.load(".data/svd_b_v.npy"),
    )
else:
    U_A, S_A, V_A = np.linalg.svd(A, full_matrices=False)
    np.save(".data/svd_a_u.npy", U_A)
    np.save(".data/svd_a_s.npy", S_A)
    np.save(".data/svd_a_v.npy", V_A)
    U_B, S_B, V_B = np.linalg.svd(B, full_matrices=False)
    np.save(".data/svd_b_u.npy", U_B)
    np.save(".data/svd_b_s.npy", S_B)
    np.save(".data/svd_b_v.npy", V_B)

In [7]:
chosen_modes = np.arange(100, 2001, 100)
reconstructed_A = np.zeros((len(chosen_modes), *A.shape))
reconstructed_B = np.zeros((len(chosen_modes), *B.shape))
relative_errors_A = np.zeros(len(chosen_modes))
relative_errors_B = np.zeros(len(chosen_modes))

for i, k in enumerate(chosen_modes):
    A_hat = U_A[:, :k] @ np.diag(S_A[:k]) @ V_A[:k, :]
    B_hat = U_B[:, :k] @ np.diag(S_B[:k]) @ V_B[:k, :]
    reconstructed_A[i] = A_hat
    reconstructed_B[i] = B_hat
    relative_errors_A[i] = np.linalg.norm(A - A_hat) / np.linalg.norm(A)
    relative_errors_B[i] = np.linalg.norm(B - B_hat) / np.linalg.norm(B)

In [ ]:
import matplotlib as mpl

mpl.rcParams["text.usetex"] = True

fig = plt.figure(dpi=300)
plt.plot(chosen_modes, relative_errors_A, '-b', label="Erro relativo A")
plt.text(1000., 0.575, r"Erro relativo de uma matriz $X$: $\frac{||X - \hat{X}||}{||X||}$")
plt.plot(chosen_modes, relative_errors_B, '-r', label="Erro relativo B")
plt.xlabel("Quantidade de componentes")
plt.ylabel("Erro relativo")
plt.legend()
plt.title("Erro relativo de reconstrução")
plt.savefig("comparativo_relative_error.png", dpi=300)
plt.show()

In [9]:
truncate_to_size = 400 
x_small = X[:truncate_to_size]

def rbf_kernel(x, y):
    gamma = 0.1
    return np.exp(-gamma * np.linalg.norm(x - y) ** 2)


def poly_kernel(x, y):
    return (1 + np.dot(x, y)) ** 2


K_rbf = np.zeros((truncate_to_size, truncate_to_size))
K_poly = np.zeros((truncate_to_size, truncate_to_size))
for i in range(truncate_to_size):
    for j in range(truncate_to_size):
        K_rbf[i, j] = rbf_kernel(x_small[i, :], x_small[j, :])
        K_poly[i, j] = poly_kernel(x_small[i, :], x_small[j, :])

In [10]:
centering_matrix = np.eye(truncate_to_size) - np.ones((truncate_to_size, truncate_to_size)) / truncate_to_size
K_rbf_centered = centering_matrix @ K_rbf @ centering_matrix
K_poly_centered = centering_matrix @ K_poly @ centering_matrix

In [11]:
# Evaluate Kernel eigenvalues

eig_kernel_rbf = np.linalg.eigvals(K_rbf)
eig_kernel_rbf_centered = np.linalg.eigvals(K_rbf_centered)
eig_kernel_poly = np.linalg.eigvals(K_poly)
eig_kernel_poly_centered = np.linalg.eigvals(K_poly_centered)

In [ ]:
cummulative_kernel_rbf = np.cumsum(eig_kernel_rbf) / np.sum(eig_kernel_rbf) * 100
plt.plot(cummulative_kernel_rbf)

In [ ]:
cummulative_kernel_rbf_centered = np.cumsum(eig_kernel_rbf_centered) / np.sum(eig_kernel_rbf_centered) * 100
plt.plot(cummulative_kernel_rbf_centered)

In [ ]:
cummulative_kernel_poly = np.cumsum(eig_kernel_poly) / np.sum(eig_kernel_poly) * 100
plt.plot(cummulative_kernel_poly)

In [ ]:
cummulative_kernel_poly_centered = np.cumsum(eig_kernel_poly_centered) / np.sum(eig_kernel_poly_centered) * 100
plt.plot(cummulative_kernel_poly_centered)